# Prática 4 — IA Generativa: textos, perguntas e recuperação de fontes

**Tempo:** 30 minutos · **Público:** iniciantes · **Ambiente:** Google Colab (CPU)

**Pergunta:** Como consultar textos de pesquisa com rastreabilidade, sem inventar evidências?

> Dados sintéticos e textos didáticos. Não representam evidência empírica nem recomendações clínicas.

### Como utilizar
Execute as células na ordem, uma de cada vez. Leia a explicação antes de clicar em ▶. Ao final, experimente alterar apenas os parâmetros indicados.

## Passo 1 — Preparar uma coleção documental

Os quatro textos são didáticos e foram escritos para o exercício, não são transcrições de fontes oficiais.

In [ ]:
documentos = {
    "DOC-01": "Saldo migratório é a diferença entre entradas e saídas em um território e período.",
    "DOC-02": "O envelhecimento populacional pode ser descrito pela participação de pessoas idosas.",
    "DOC-03": "Projeções populacionais dependem de hipóteses sobre fecundidade, mortalidade e migração.",
    "DOC-04": "A taxa bruta de natalidade relaciona nascidos vivos à população média no período."
}
print("Documentos disponíveis:", len(documentos))

## Passo 2 — Transformar textos em números

TF-IDF representa termos importantes em cada texto; nesta versão não utilizamos embeddings neurais.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

identificadores = list(documentos)
textos = list(documentos.values())

vetorizador = TfidfVectorizer(strip_accents="unicode")
matriz = vetorizador.fit_transform(textos)
print("Formato da matriz:", matriz.shape)

## Passo 3 — Fazer uma pergunta

A busca compara os termos da pergunta com os termos dos documentos.

In [ ]:
pergunta = "Quais fatores entram nas projeções populacionais?"
vetor_pergunta = vetorizador.transform([pergunta])

similaridades = cosine_similarity(vetor_pergunta, matriz)[0]
ordem = similaridades.argsort()[::-1]

for posicao in ordem[:2]:
    print(identificadores[posicao], "—", textos[posicao])

## Passo 4 — Produzir uma resposta rastreável

Nesta etapa demonstramos a recuperação, não uma geração real por LLM. Assim, tudo funciona sem credenciais ou downloads.

In [ ]:
melhor = ordem[0]
if similaridades[melhor] > 0:
    print("Trecho recuperado:")
    print(f"[{identificadores[melhor]}] {textos[melhor]}")
else:
    print("Não há evidência textual suficiente no acervo.")

## Passo 5 — Criar um prompt para um LLM

O texto abaixo ilustra como usar a fonte recuperada. Não enviamos dados a serviços externos.

In [ ]:
fonte = f"[{identificadores[melhor]}] {textos[melhor]}"

prompt = f"""
Responda à pergunta usando apenas a fonte abaixo.
Cite o identificador do documento.
Se não houver evidência, diga que não consta na fonte.

Pergunta: {pergunta}

Fonte:
{fonte}
"""
print(prompt)

## Agora é sua vez

Troque a pergunta por 'O que é saldo migratório?' e depois por 'Qual era a população brasileira em 2022?'. Explique por que a segunda pergunta não deve receber um número inventado.

## Antes de usar dados reais

Documente a origem, a qualidade, o período, a unidade de observação e as condições de uso dos dados. Uma boa previsão ou resposta automática não demonstra causalidade nem dispensa validação.